# 00 · Source and relational schema

Operations wants to know where orders miss their promised date and how lateness relates to customer feedback. This first notebook maps the nine Olist CSVs and identifies the join grain needed for both questions. The displayed data comes from the verified local pipeline. Run the [reproduction steps](../README.md#reproduce) to execute the cells.

[Notebook guide](../README.md#notebook-guide) · [Next](01_data_quality_and_sql.ipynb)

In [1]:
import json
import sqlite3
from contextlib import closing
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "src/olist_analytics").is_dir(), (
    "Open from the project root or notebooks directory."
)
REPORTS = ROOT / "reports"
TABLES = REPORTS / "tables"


DB = ROOT / "data/processed/olist.sqlite3"
DB_URI = DB.as_uri() + "?mode=ro"

## What does each source row represent?

Review the recorded import counts against each table’s role. Orders carry promise and delivery timestamps. Items, payments and reviews can contribute several rows to the same order. Customer location joins through customer_id. Customer clustering uses customer_unique_id across orders.

In [2]:
quality = json.loads((REPORTS / "data_quality.json").read_text())
audit_tables = pd.DataFrame.from_dict(
    {
        name: info
        for name, info in quality.items()
        if isinstance(info, dict) and "raw_rows" in info
    },
    orient="index",
)[["raw_rows", "clean_rows", "exact_duplicate_rows_removed"]]
manifest = json.loads((REPORTS / "source_manifest.json").read_text())
print(f"Source archive SHA-256 {manifest['archive_sha256']}")
display(audit_tables)

Source archive SHA-256 967e41e04fc306fe604e2a693f488995a8b41e5047418f8a5c8e4abd6deca784


,raw_rows,clean_rows,exact_duplicate_rows_removed
customers,99441,99441,0
geolocation,1000163,738332,261831
order_items,112650,112650,0
order_payments,103886,103886,0
order_reviews,99224,99224,0
orders,99441,99441,0
products,32951,32951,0
sellers,3095,3095,0
category_translation,71,71,0


## How could a join inflate counts?

Count repeated order keys in each child table with GROUP BY and HAVING. These are normal relational multiplicities. Joining several children directly on order_id would form combinations of their rows.

In [3]:
with closing(sqlite3.connect(DB_URI, uri=True)) as connection:
    for table in ("order_items", "order_payments", "order_reviews"):
        sql = f"""
        SELECT COUNT(*) AS orders_with_multiple_rows
        FROM (
            SELECT order_id
            FROM {table}
            GROUP BY order_id
            HAVING COUNT(*) > 1
        )
        """
        print(sql.strip())
        display(pd.read_sql_query(sql, connection).assign(source=table))

SELECT COUNT(*) AS orders_with_multiple_rows
        FROM (
            SELECT order_id
            FROM order_items
            GROUP BY order_id
            HAVING COUNT(*) > 1
        )


,orders_with_multiple_rows,source
0,9803,order_items


SELECT COUNT(*) AS orders_with_multiple_rows
        FROM (
            SELECT order_id
            FROM order_payments
            GROUP BY order_id
            HAVING COUNT(*) > 1
        )


,orders_with_multiple_rows,source
0,2961,order_payments


SELECT COUNT(*) AS orders_with_multiple_rows
        FROM (
            SELECT order_id
            FROM order_reviews
            GROUP BY order_id
            HAVING COUNT(*) > 1
        )


,orders_with_multiple_rows,source
0,547,order_reviews


Child tables need a defined reduction before the order join. Geolocation can also repeat at postal-prefix grain and has no role in the state join. The next notebook checks which records and fields can support an order-level model.

[Notebook guide](../README.md#notebook-guide) · [Next](01_data_quality_and_sql.ipynb)